# Train Super Mario Bros in Google Colab

Run the cells from top to bottom. Before starting, choose **Runtime → Change runtime type → GPU**. The notebook will ask you to mount Google Drive and upload `trainmario.py` plus `evaluatemario.py`. Model checkpoints and logs are stored in Drive so they survive runtime resets.

Training defaults to phase 1 (level 1-1, 2 million timesteps). A useful first run can be stopped after the first saved checkpoints; continue later by setting a checkpoint path in the resume cell. Colab runtimes are temporary, so periodically check that checkpoints are appearing in Drive.

In [3]:
%pip install -q "stable-baselines3[extra]" gymnasium gym-super-mario-bros opencv-python-headless tensorboard

Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Choose Runtime → Change runtime type → GPU, reconnect, and run this cell again.')
print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.13.0+cpu
CUDA available: False


RuntimeError: No GPU detected. Choose Runtime → Change runtime type → GPU, reconnect, and run this cell again.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

PROJECT_DIR = Path('/content/super_mario')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()  # Select trainmario.py and evaluatemario.py from your project folder.
required = {'trainmario.py', 'evaluatemario.py'}
missing = required - set(uploaded)
if missing:
    raise FileNotFoundError(f'Upload both project scripts. Missing: {sorted(missing)}')
for filename in required:
    shutil.move(filename, PROJECT_DIR / filename)
print('Project scripts ready in', PROJECT_DIR)

In [ ]:
import os, sys

DRIVE_TRAIN_DIR = '/content/drive/MyDrive/super_mario_training'
os.makedirs(DRIVE_TRAIN_DIR, exist_ok=True)
os.environ['MARIO_TRAIN_DIR'] = DRIVE_TRAIN_DIR
os.environ['MARIO_RENDER'] = '0'  # Colab cannot display the emulator's desktop window.
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Training outputs will be saved to:', DRIVE_TRAIN_DIR)

## Check the game environment

This creates one environment, checks its image observation, takes one no-op step, then closes it. Fix any error here before starting a long training run.

In [ ]:
from trainmario import make_mario_env, PHASES, PHASE
env = make_mario_env(PHASES[PHASE]['env_id'])()
obs, info = env.reset()
obs, reward, terminated, truncated, info = env.step(0)
print('Observation:', obs.shape, obs.dtype, '| reward:', reward, '| x position:', info.get('x_pos'))
env.close()

## Resume from the latest saved model

This cell searches your mounted Drive for the most recently saved checkpoint, interrupted model, or final model and selects it automatically. If training is new and Drive has no model yet, it starts a new run. After a graceful stop, wait until the training cell prints `Saved resume point` before closing the runtime. Colab also writes a periodic checkpoint every 100,000 steps in case the runtime disconnects unexpectedly.

In [ ]:
from pathlib import Path

# Optional: put a specific .zip path here to override automatic selection.
RESUME_FROM = ''
if not RESUME_FROM.strip():
    saved_models = Path(DRIVE_TRAIN_DIR) / 'saved_models'
    patterns = ('*/checkpoints/mario_*_steps.zip', '*/interrupted_model.zip', '*/final_model.zip', '*/best_model.zip')
    candidates = [p for pattern in patterns for p in saved_models.glob(pattern)]
    RESUME_FROM = str(max(candidates, key=lambda p: p.stat().st_mtime)) if candidates else ''
if RESUME_FROM.strip():
    os.environ['MARIO_LOAD_MODEL_PATH'] = RESUME_FROM.strip()
    print('Resuming from:', RESUME_FROM)
else:
    os.environ.pop('MARIO_LOAD_MODEL_PATH', None)
    print('No saved model found in Drive; training will start from step 0.')

## Start training

Run this cell when the GPU, Drive, script upload, and environment check all succeeded. To pause safely, interrupt this cell and wait for `Saved resume point` to appear. Model files are written under `super_mario_training/saved_models` in Drive.

In [ ]:
%run trainmario.py

## Evaluate the latest saved model

After training has saved a best model, run this cell for episode summaries. Colab cannot show the emulator's normal desktop window; set `MARIO_RENDER=1` and run evaluation on your local PC if you want to watch the game.

In [ ]:
!python -u evaluatemario.py

In [ ]:
!nvidia-smi